In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("sources", "yellow,green,zones")
dbutils.widgets.text("start_month", "2025-01")
dbutils.widgets.text("end_month", "2025-12")

In [0]:
# Auto reload when code in src/ has changes

%load_ext autoreload
%autoreload 2

In [0]:
# import src/ module and sources.yml

import os, sys
import yaml

src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from nyctaxi.landing import land_source

with open("../config/sources.yml") as f:
    config = yaml.safe_load(f)

In [0]:
# create schema and volume if not exists and set catalog

catalog = dbutils.widgets.get("catalog")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.landing")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.landing.raw_files")

In [0]:
sources = [s.strip() for s in dbutils.widgets.get("sources").split(",") if s.strip()]
start_month = dbutils.widgets.get("start_month")
end_month = dbutils.widgets.get("end_month")

# raise error if theres something wrong with the sources name
unknown = [name for name in sources if name not in config["sources"]]
if unknown:
    raise ValueError(f"No source in sources.yml: {unknown}")

base_dir = f"/Volumes/{catalog}/landing/raw_files"

for name in sources:
    cfg = config["sources"][name]                 # cấu hình của nguồn này
    target_dir = f"{base_dir}/{name}"             # mỗi nguồn một thư mục
    print(f"--- {name} ---")
    land_source(cfg, start_month, end_month, target_dir)

In [0]:
for name in sources:
    target_dir = f"{base_dir}/{name}"
    files = sorted(dbutils.fs.ls(target_dir), key=lambda f: f.name)
    total_mb = sum(f.size for f in files) / 1024**2
    print(f"{name}: {len(files)} file, {total_mb:.1f} MB")
    for f in files:
        print(f"   {f.name}: {f.size / 1024**2:.1f} MB")